# Purpose:
- Compare cell matching between
    - ROICat
    - Registering to cortical z-stack

In [14]:
from pathlib import Path
import pandas as pd
import numpy as np

In [228]:
# Load fov to cortical z-stack matching table
using_stack_df = pd.read_csv('/root/capsule/scratch/glm/thyme_fov_to_czstack_match_table.csv')
using_stack_df['session_key'] = using_stack_df.session_name.apply(lambda x: '_'.join(x.split('_')[1:3]))
using_stack_df['roi_name'] = using_stack_df.apply(lambda x: f'{"_".join(x.session_key.split("_")[-2:])}_{x.plane}_{x.session_roi_id:04}', axis=1)

In [229]:
# Filter duplicated rois based on iou
temp_series = using_stack_df.groupby('roi_name').size()
temp_series = temp_series[temp_series > 1]
print(f'Expected number of rows to remove: {temp_series.sum() - len(temp_series)}')
# print(temp_series.max())
duplicated_roi_names = temp_series.index.values
print(f'Number of duplicated ROIs: {len(duplicated_roi_names)}')
duplicated_roi_df = using_stack_df[using_stack_df.roi_name.isin(duplicated_roi_names)].copy()
print(len(duplicated_roi_df))
num_rows_to_remove = len(duplicated_roi_df) - len(duplicated_roi_names)
print(f'Number of rows to remove: {num_rows_to_remove}')
print(f'Expected rows after removal: {len(using_stack_df) - num_rows_to_remove}')
inds_to_remove = duplicated_roi_df.index.values
duplicated_roi_df.sort_values('max_iou', ascending=False, inplace=True)
duplicated_roi_df.drop_duplicates('roi_name', keep='first', inplace=True)
print(len(duplicated_roi_df))
inds_to_keep = duplicated_roi_df.index.values
inds_to_remove = np.setdiff1d(inds_to_remove, inds_to_keep)
using_stack_df.drop(inds_to_remove, inplace=True)
print(f'Rows after removal: {len(using_stack_df)}')
assert (using_stack_df.groupby('roi_name').size()>1).sum() == 0

Expected number of rows to remove: 97
Number of duplicated ROIs: 96
193
Number of rows to remove: 97
Expected rows after removal: 9124
96
Rows after removal: 9124


In [202]:
roicat_df = pd.read_pickle('/root/capsule/scratch/glm/roicat_df_736963.pkl')
print(len(roicat_df))
valid_roicat_df = roicat_df.query('valid_roi').copy()
print(len(valid_roicat_df))
df_list = []
for fn in valid_roicat_df.fov_name.unique():
    temp_df = valid_roicat_df[valid_roicat_df.fov_name == fn].copy()
    max_ucid = temp_df.ucid.max()
    single_roi_names = temp_df.query('matched==False').index.values
    for roi_name in single_roi_names:
        max_ucid += 1
        temp_df.loc[roi_name, 'ucid'] = max_ucid
        temp_df.loc[roi_name, 'unique_cell_name'] = f'{fn}_{max_ucid:04}'
    df_list.append(temp_df)
valid_roicat_df = pd.concat(df_list)
print(valid_roicat_df.unique_cell_name.nunique())
print(valid_roicat_df.groupby('unique_cell_name').size().sum())

7069
5226
934
5226


In [231]:
# Check the number of rows in using_stack_df that are in valid_roicat_df
session_keys_in_roicat = valid_roicat_df.session_key.unique()
nis_using_stack_df = using_stack_df[using_stack_df.session_key.isin(session_keys_in_roicat)].copy() # natural image sessions
print(len(nis_using_stack_df))
print(len(nis_using_stack_df)/len(valid_roicat_df))
# using_stack_df is not fully filtered by valid rois
# (so there are some rois that are not in valid_roicat_df)

3978
0.7611940298507462


In [237]:
# How about per session?
# Look at it after merging
reduced_valid_roicat_df = valid_roicat_df.reset_index()[['roi_name', 'unique_cell_name', 'session_key']].copy()
reduced_using_stack_df = using_stack_df[['roi_name', 'cz_stack_id']].copy()
comp_df = reduced_valid_roicat_df.merge(reduced_using_stack_df,
                                        on='roi_name',
                                        how='left')
comp_df


,roi_name,unique_cell_name,session_key,cz_stack_id
0,736963_2024-08-09_VISp_0_0004,VISp_0_0013,736963_2024-08-09,NaN
1,736963_2024-08-09_VISp_0_0005,VISp_0_0088,736963_2024-08-09,379.0
2,736963_2024-08-09_VISp_0_0006,VISp_0_0092,736963_2024-08-09,439.0
3,736963_2024-08-09_VISp_0_0007,VISp_0_0101,736963_2024-08-09,363.0
4,736963_2024-08-09_VISp_0_0008,VISp_0_0098,736963_2024-08-09,361.0
...,...,...,...,...
5221,736963_2024-08-06_VISp_7_0093,VISp_7_0071,736963_2024-08-06,NaN
5222,736963_2024-08-06_VISp_7_0095,VISp_7_0084,736963_2024-08-06,789.0
5223,736963_2024-08-06_VISp_7_0096,VISp_7_0088,736963_2024-08-06,NaN
5224,736963_2024-08-06_VISp_7_0097,VISp_7_0080,736963_2024-08-06,NaN


In [242]:
# proportion dropped when matching to cortical z-stack
prop_missing_from_stack = comp_df.cz_stack_id.isna().mean()
print(f'Proportion of ROIs missing from cortical z-stack: {prop_missing_from_stack}')
num_rois_per_session = comp_df.groupby('session_key').size()
num_missing_rois_per_session = comp_df[comp_df.cz_stack_id.isna()].groupby('session_key').size()
prop_missing_rois_per_session = num_missing_rois_per_session/num_rois_per_session
prop_missing_rois_per_session


Proportion of ROIs missing from cortical z-stack: 0.34998086490623803


session_key
736963_2024-07-24    0.344538
736963_2024-07-26    0.328829
736963_2024-07-29    0.329218
736963_2024-07-30    0.361377
736963_2024-08-01    0.347458
736963_2024-08-05    0.329621
736963_2024-08-06    0.372745
736963_2024-08-07    0.329384
736963_2024-08-09    0.334052
736963_2024-08-12    0.364017
736963_2024-08-13    0.397661
dtype: float64

In [255]:
final_comp_df = comp_df[comp_df.cz_stack_id.notna()].copy()
print(f'Num unique cell names from roicat: {final_comp_df.unique_cell_name.nunique()}')
print(f'Num rois from cortical z-stack: {final_comp_df.cz_stack_id.nunique()}')


Num unique cell names from roicat: 500
Num rois from cortical z-stack: 436


In [256]:
final_comp_df.groupby('unique_cell_name')['cz_stack_id'].nunique().value_counts()

cz_stack_id
1    487
2     13
Name: count, dtype: int64

In [252]:
final_comp_df.groupby('cz_stack_id')['unique_cell_name'].nunique().value_counts()

unique_cell_name
1     374
2      58
3       2
6       1
11      1
Name: count, dtype: int64

In [260]:
from_zstack_to_roicat = final_comp_df.groupby('cz_stack_id')['unique_cell_name'].nunique()
from_zstack_to_roicat[from_zstack_to_roicat > 2]

cz_stack_id
346.0     6
532.0    11
626.0     3
684.0     3
Name: unique_cell_name, dtype: int64

In [262]:
np.sort(final_comp_df.session_key.unique())

array(['736963_2024-07-24', '736963_2024-07-26', '736963_2024-07-29',
       '736963_2024-07-30', '736963_2024-08-01', '736963_2024-08-05',
       '736963_2024-08-06', '736963_2024-08-07', '736963_2024-08-09',
       '736963_2024-08-12', '736963_2024-08-13'], dtype=object)

In [264]:
# What if I reduce to the ones in FNN session?
fnn_session_keys = ['736963_2024-08-06', '736963_2024-08-07', '736963_2024-08-09']
fnn_comp_df = final_comp_df[final_comp_df.session_key.isin(fnn_session_keys)].copy()
display(fnn_comp_df.groupby('unique_cell_name')['cz_stack_id'].nunique().value_counts())
display(fnn_comp_df.groupby('cz_stack_id')['unique_cell_name'].nunique().value_counts())


cz_stack_id
1    404
2      5
Name: count, dtype: int64

unique_cell_name
1    335
2     38
3      1
Name: count, dtype: int64

In [270]:
# save the final comp_df
save_dir = Path('/root/capsule/scratch/roi_matching_qc')
save_dir.mkdir(exist_ok=True, parents=True)
final_comp_df.to_csv(save_dir /'736963_roi_matching_comp_df_v1.csv', index=False)

# Conclusion:
- There is multi matching examples.
- Check them visually.
- In a separate capsule. (CTL QC)
    - I also need the result from this capsule (or this notebook)

In [6]:
merged_df = pd.read_csv('/root/capsule/scratch/glm/Thyme_merged_roi_ids_with_cluster_ids.csv')